In [1]:
import cv2
import numpy as np
import math

In [2]:
import cv2
import numpy as np
import math

class HSVChangeDetector:
    def __init__(self, threshold=27.0, weights=[1.0, 1.0, 1.0, 0.0]):
        self.threshold = threshold
        self.weights = weights
        self._last_hue =None
        
        self._last_sat= None

        self._last_lum= None
        self._last_edges =None
        self._kernel =None

    def process_frame(self, frame_num, bgr_frame):
        hsv =cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2HSV)
        hue, sat, lum=cv2.split(hsv)

        if self._kernel is None:
            h, w = lum.shape
            size = 4 + round(math.sqrt(w * h) / 192)
            if size % 2 == 0:
                size += 1
            self._kernel= np.ones((size, size), np.uint8)

        sigma = 1.0 / 3.0
        median =np.median(lum)
        low = int(max(0, (1.0 - sigma) * median))
        high= int(min(255, (1.0 + sigma) * median))
        edges =cv2.dilate(cv2.Canny(lum, low, high), self._kernel)

        if self._last_lum is None:
            self._last_hue, self._last_sat, self._last_lum, self._last_edges = hue, sat, lum, edges
            return 0.0

        def mpd(a, b):
            return np.sum(np.abs(a.astype(np.int32) - b.astype(np.int32))) / float(a.shape[0] * a.shape[1])

        dh= mpd(hue, self._last_hue)
        ds =mpd(sat, self._last_sat)
        dl= mpd(lum, self._last_lum)
        de= mpd(edges, self._last_edges)

        score =(self.weights[0]*dh +self.weights[1]*ds+ self.weights[2]*dl + self.weights[3]*de) / sum(abs(w) for w in self.weights)

        self._last_hue,self._last_sat, self._last_lum,self._last_edges = hue, sat, lum, edges
        return score


In [3]:
class FlashFilter:
    def __init__(self, mode="SUPPRESS", length=15):
        self.mode = mode
        self.length = length
        self._last_above = None
        self._merge_enabled = False
        self._merge_triggered = False
        self._merge_start = None

    def filter(self, frame_num, above_threshold):
        if self._last_above is None:
            self._last_above = frame_num
        if self.mode == "MERGE":
            return self._merge(frame_num, above_threshold)
        return self._suppress(frame_num, above_threshold)

    def _suppress(self, frame_num, above_threshold):
        min_len = (frame_num - self._last_above) >= self.length
        if not (above_threshold and min_len):
            return []
        self._last_above = frame_num
        return [frame_num]

    def _merge(self, frame_num, above_threshold):
        min_len = (frame_num - self._last_above) >= self.length
        if above_threshold:
            self._last_above = frame_num
        if self._merge_triggered:
            merged = self._last_above - self._merge_start
            if min_len and not above_threshold and merged >= self.length:
                self._merge_triggered = False
                return [self._last_above]
            return []
        if not above_threshold:
            return []
        if min_len:
            self._merge_enabled = True
            return [frame_num]
        if self._merge_enabled:
            self._merge_triggered = True
            self._merge_start = frame_num
        return []


In [ ]:
import cv2
import numpy as np

video_path = "./dataset/videos/tears_of_steel_1080p.mp4"
cap = cv2.VideoCapture(video_path)

detector = HSVChangeDetector(threshold=30.0)
ff = FlashFilter(mode="MERGE", length=15)

frame_idx = 0
scores = []
boundaries = [0]

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (320, 180))  # downscale for speed
    score = detector.process_frame(frame_idx, frame)
    scores.append((frame_idx, score))

    cuts = ff.filter(frame_idx, score >= detector.threshold)
    boundaries.extend(cuts)

    frame_idx += 1

total_frames = frame_idx
boundaries.append(total_frames - 1)

scenes = []
for i in range(len(boundaries) - 1):
    start = boundaries[i]
    end = boundaries[i+1] - 1
    scenes.append((i+1, start, end))

print("Detected scenes (scene_num, start, end):")
for s in scenes:
    print(s[0], s[1], s[2])

cap.release()


In [ ]:
import importlib
import sys
sys.path.insert(0, ".")
import utils
importlib.reload(utils)
from utils import calculate_interval_metric

gt_txt = "./dataset/videos/tears_of_steel_1080p_scenes.txt"
gt_intervals = []
with open(gt_txt) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) == 2:
            gt_intervals.append((int(parts[0]), int(parts[1])))

our_intervals = [(s[1], s[2]) for s in scenes]

print(f"GT:   {len(gt_intervals)} scenes")
print(f"Ours: {len(our_intervals)} scenes")
print()
for m in ['precision', 'recall', 'f1', 'iou']:
    score = calculate_interval_metric([gt_intervals], [our_intervals], m)
    print(f"  {m:12s}: {score:.3f}")


In [ ]:
our_intervals = [(s[1], s[2]) for s in scenes]

print(f"GT:      {len(gt_intervals)} scenes")
print(f"Ours:    {len(our_intervals)} scenes")
print()

print(f"{'metric':12s} {'Ours':>10} {'Package':>10}")
for m in ['precision', 'recall', 'f1', 'iou']:
    ours_score = calculate_interval_metric([gt_intervals], [our_intervals], m)
    print(f"{m:12s} {ours_score:>10.3f}")
